In [1]:
import pandas as pd
import spacy
from spacy.tokens import DocBin
from spacy.util import filter_spans
from sklearn.model_selection import train_test_split
import random
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('Data/ner_bioes.csv')
print(df.head(10))
print(f"\nShape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

                                  sentence       token    tag
0  Apple launches new iPhone in California       Apple  S-ORG
1  Apple launches new iPhone in California    launches      O
2  Apple launches new iPhone in California         new      O
3  Apple launches new iPhone in California      iPhone      O
4  Apple launches new iPhone in California          in      O
5  Apple launches new iPhone in California  California  S-LOC
6                 Elon Musk founded SpaceX        Elon  B-PER
7                 Elon Musk founded SpaceX        Musk  E-PER
8                 Elon Musk founded SpaceX     founded      O
9                 Elon Musk founded SpaceX      SpaceX  S-ORG

Shape: (100, 3)

Columns: ['sentence', 'token', 'tag']


In [3]:
sentences = []

for text, group in df.groupby('sentence', sort=False):
    sentences.append({
        'sentence': text,
        'tokens': group['token'].tolist(),
        'tags': group['tag'].tolist()
    })

print(f"Total sentences: {len(sentences)}")
print("\nFirst 2 entries:")
for s in sentences[:2]:
    print(s)

Total sentences: 22

First 2 entries:
{'sentence': 'Apple launches new iPhone in California', 'tokens': ['Apple', 'launches', 'new', 'iPhone', 'in', 'California'], 'tags': ['S-ORG', 'O', 'O', 'O', 'O', 'S-LOC']}
{'sentence': 'Elon Musk founded SpaceX', 'tokens': ['Elon', 'Musk', 'founded', 'SpaceX'], 'tags': ['B-PER', 'E-PER', 'O', 'S-ORG']}


In [4]:
def validate_and_fix_bioes(sentences):
    """
    Validate BIOES tag sequences and fix invalid transitions.
    Returns the cleaned list of sentences and a report.
    """
    fixed_sentences = []
    total_warnings = 0
    sentences_with_issues = 0

    for s in sentences:
        tags = s['tags'].copy()
        has_issue = False
        n = len(tags)

        for i in range(n):
            current_tag = tags[i]
            next_tag = tags[i+1] if i + 1 < n else "O"
            
            curr_prefix = current_tag[0] if current_tag != "O" else "O"
            curr_label = current_tag[2:] if current_tag != "O" else ""
            
            next_prefix = next_tag[0] if next_tag != "O" else "O"
            next_label = next_tag[2:] if next_tag != "O" else ""

            invalid = False
            if curr_prefix == "B" and next_prefix not in ["I", "E"]:
                invalid = True
            elif curr_prefix == "I" and next_prefix not in ["I", "E"]:
                invalid = True
            elif curr_prefix in ["B", "I"] and curr_label != next_label:
                invalid = True
            elif curr_prefix in ["O", "E", "S"] and next_prefix in ["I", "E"]:
                invalid = True

            if invalid:
                has_issue = True
                total_warnings += 1
                # Stratégie de correction : on repasse le tag problématique à "O"
                tags[i] = "O"

        fixed_sentences.append({
            'sentence': s['sentence'],
            'tokens': s['tokens'],
            'tags': tags
        })
        if has_issue:
            sentences_with_issues += 1

    print(f"\n✅ Validation complete.")
    print(f"   Sentences with issues: {sentences_with_issues}")
    print(f"   Total warnings issued: {total_warnings}")
    return fixed_sentences


sentences_clean = validate_and_fix_bioes(sentences)


✅ Validation complete.
   Sentences with issues: 0
   Total warnings issued: 0


In [5]:
def bioes_to_token_spans(sentences_clean):
    """
    Étape 1: Convertit le format BIOES en index basés sur les TOKENS
    Format attendu: [(start_token, end_token, label)]
    """
    token_spans_data = []
    
    for s in sentences_clean:
        tokens = s['tokens']
        tags = s['tags']
        token_entities = []
        
        start_token_idx = None
        current_label = None
        
        for i, tag in enumerate(tags):
            prefix = tag[0] if tag != "O" else "O"
            label = tag[2:] if tag != "O" else ""
            
            if prefix == "S":
                token_entities.append((i, i + 1, label))
            elif prefix == "B":
                start_token_idx = i
                current_label = label
            elif prefix == "E" and start_token_idx is not None:
                token_entities.append((start_token_idx, i + 1, current_label))
                start_token_idx = None
                current_label = None
            elif prefix == "O":
                start_token_idx = None
                current_label = None
                
        token_spans_data.append({
            'sentence': s['sentence'],
            'tokens': tokens,
            'token_entities': token_entities
        })
    return token_spans_data


def map_to_char_spans(token_spans_data):
    """
    Étape 2: Mappe les index de tokens en index de CARACTÈRES globaux
    Format attendu: (text, {"entities": [(start_char, end_char, label)]})
    """
    training_data = []
    
    for item in token_spans_data:
        text = item['sentence']
        tokens = item['tokens']
        token_entities = item['token_entities']
        char_entities = []
        
        for start_tok, end_tok, label in token_entities:

            start_char = text.find(tokens[start_tok])
            

            last_tok_start = text.find(tokens[end_tok - 1], start_char)
            end_char = last_tok_start + len(tokens[end_tok - 1])
            
            if start_char != -1 and last_tok_start != -1:
                char_entities.append((start_char, end_char, label))
                
        # Format demandé par la Task 4 {"entities": [...]}
        training_data.append((text, {"entities": char_entities}))
        
    return training_data

token_spans = bioes_to_token_spans(sentences_clean)
training_data = map_to_char_spans(token_spans)

In [6]:
print("Entity spot-check:")
for text, annot in training_data[:5]:
    for (start, end, label) in annot['entities']:
        entity_text = text[start:end]
        print(f"  [{label}] '{entity_text}'  (chars {start}:{end})")

Entity spot-check:
  [ORG] 'Apple'  (chars 0:5)
  [LOC] 'California'  (chars 29:39)
  [PER] 'Elon Musk'  (chars 0:9)
  [ORG] 'SpaceX'  (chars 18:24)
  [ORG] 'Google'  (chars 0:6)
  [LOC] 'London'  (chars 23:29)
  [ORG] 'Microsoft'  (chars 0:9)
  [LOC] 'Berlin'  (chars 29:35)
  [LOC] 'Paris'  (chars 0:5)


In [7]:
from spacy.tokens import DocBin
from spacy.util import filter_spans
from tqdm import tqdm

In [8]:
nlp = spacy.blank("en")
doc_bin = DocBin()
skipped = 0
added = 0

for text, annot in tqdm(training_data):
    entities = annot['entities']
    
    doc = nlp.make_doc(text)
    spans = []
    
    for start, end, label in entities:
        span = doc.char_span(start, end, label=label, alignment_mode="contract")
        if span is not None:
            spans.append(span)
        else:
            skipped += 1
            
    filtered_spans = filter_spans(spans)
    doc.ents = filtered_spans
    doc_bin.add(doc)
    added += 1

doc_bin.to_disk("train.spacy")
print(f"\nSaved DocBin: {added} docs added, {skipped} entities skipped.")

100%|██████████| 22/22 [00:00<00:00, 7341.45it/s]


Saved DocBin: 22 docs added, 0 entities skipped.


In [9]:
SEED = 42
random.seed(SEED)

data_shuffled = training_data.copy()
random.shuffle(data_shuffled)

n = len(data_shuffled)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_data = data_shuffled[:train_end]
valid_data = data_shuffled[train_end:valid_end]
test_data = data_shuffled[valid_end:]

def build_docbin(data, nlp):
    db = DocBin()
    for text, annot in data:
        doc = nlp.make_doc(text)
        spans = []
        for start, end, label in annot['entities']:
            span = doc.char_span(start, end, label=label, alignment_mode="contract")
            if span is not None:
                spans.append(span)
        doc.ents = filter_spans(spans)
        db.add(doc)
    return db

train_bin = build_docbin(train_data, nlp)
train_bin.to_disk("train.spacy")

valid_bin = build_docbin(valid_data, nlp)
valid_bin.to_disk("valid.spacy")

test_bin = build_docbin(test_data, nlp)
test_bin.to_disk("test.spacy")

print(f"Split complet :")
print(f"   - Train (70%)      : {len(train_data)} phrases")
print(f"   - Validation (15%) : {len(valid_data)} phrases")
print(f"   - Test (15%)       : {len(test_data)} phrases")

Split complet :
   - Train (70%)      : 15 phrases
   - Validation (15%) : 3 phrases
   - Test (15%)       : 4 phrases
